## CLIP ASIN-Level Embedding Extraction

**Provenance note.** This notebook originally contained several extraction variants accumulated across analyses (SBERT text embeddings, OpenCLIP, brand-level aggregation with hierarchical imputation). Only the chunk below was used for the **final ASIN-level analysis**: it produced `asin_features_for_blp_clip.csv` (167 ASINs; 155 with both modalities, 12 text-only), which feeds the joint PCA (`asin_joint_pca.R`) that generates `data/asin_joint_pcs_complete.csv` used by all R estimation scripts. The unused variants have been removed.

Verification anchors (all confirmed against the shipped data):
- the 4 APAGARD ASINs carry `chosen_img_url` equal to the first working URL in `ASIN_IMAGE_OVERRIDE`;
- the column order (`asin_img_pc1, asin_text_pc1, asin_img_pc2, ...`) matches the interleaved write loop below;
- modality-specific PCA fitted on valid rows only ⇒ 12 text-only ASINs have NaN image PCs, handled downstream by text-only projection.

In [ ]:
# ============================================================
# CLIP ASIN-LEVEL EMBEDDING EXTRACTION  (used for the final ASIN x quarter pipeline)
# ============================================================
# This is the extraction step that produced asin_features_for_blp_clip.csv,
# the input to the joint PCA (asin_joint_pca.R) that generates
# data/asin_joint_pcs_complete.csv used by all estimation scripts.
#
# INPUT: CSV or PARQUET with at least:
#   - asin
#   - brand_grouped_50
#   - image_url   (can be empty)
#   - title
#   - product_benefit
#
# OUTPUT:
#   asin_features_for_blp_clip.csv / .parquet
#     one row per ASIN: txn_count, has_img, has_text, chosen_img_url,
#     asin_img_pc1..5, asin_text_pc1..5
#
# Notes:
# - Model: openai/clip-vit-base-patch32 (HuggingFace transformers), 512-d projections.
# - Image and text embeddings are L2-normalised.
# - ASIN_IMAGE_OVERRIDE: rule-based multi-URL fallback for 4 APAGARD ASINs whose
#   catalogue image URLs were dead; each list is tried in order until one downloads.
# - PCA (5 components per modality) is fitted ONLY on ASINs with a valid embedding
#   for that modality; ASINs without an image get NaN image PCs here and are later
#   projected into the joint PC space from text alone (see thesis, Data section).
# ============================================================

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# ---- CONFIG ----
INPUT_PATH = "/content/asin_master_for_embeddings.parquet"  # <-- change if needed
OUTPUT_DIR = "clip_brand_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ASIN_COL = "asin"
BRAND_COL = "brand_grouped_50"
IMG_URL_COL = "image_url"
TITLE_COL = "title"
BENEFIT_COL = "product_benefit"

N_PCS = 5
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

# ---- OPTIONAL: ASIN -> multi-image URL override (APAGARD fix) ----
ASIN_IMAGE_OVERRIDE = {
    "B0016GCZQO": [
        "https://m.media-amazon.com/images/I/71b-CXWo-tL._AC_SL1500_.jpg",
        "https://m.media-amazon.com/images/I/71bQTon4kYL._AC_SL1500_.jpg",
        "https://m.media-amazon.com/images/I/71laQ1g2ncL._AC_SL1050_.jpg",
    ],
    "B0016GCZSC": [
        "https://m.media-amazon.com/images/I/51Y3Ra-Dy3L._AC_SY879_.jpg",
        "https://m.media-amazon.com/images/I/61Gf4MpHvYL._AC_SX679_.jpg",
        "https://m.media-amazon.com/images/I/61S2ghjI8HL._AC_SX679_.jpg",
        "https://m.media-amazon.com/images/I/81E3qcTvrXL._AC_SY879_.jpg",
    ],
    "B0016GHKLO": [
        "https://m.media-amazon.com/images/I/51nwL8TitxL._SX425_.jpg",
        "https://m.media-amazon.com/images/I/61avYlYMDKL._SX425_.jpg",
    ],
    "B06X3Q9B5X": [
        "https://m.media-amazon.com/images/I/51ub-L7NoGL._AC_SY300_SX300_QL70_ML2_.jpg",
        "https://m.media-amazon.com/images/I/514Wc-7gPtL._AC_SX522_.jpg",
        "https://m.media-amazon.com/images/I/61FBdlVgqmL._AC_SX522_.jpg",
        "https://m.media-amazon.com/images/I/41gGzMC5AcL._AC_.jpg",
    ],
}

# ---- DEPS ----
# If you're on Colab, you may need:
# !pip -q install transformers torch torchvision pillow requests tqdm scikit-learn

import requests
from PIL import Image
from io import BytesIO

import torch
from transformers import CLIPProcessor, CLIPModel

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# -----------------------------
# Helpers
# -----------------------------
def clean_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def first_nonempty(series):
    for v in series:
        v = clean_str(v)
        if v != "":
            return v
    return ""

def read_any(path: str) -> pd.DataFrame:
    lp = path.lower()
    if lp.endswith(".parquet"):
        return pd.read_parquet(path)
    if lp.endswith(".csv"):
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path} (use .csv or .parquet)")

def download_image(url_or_urls, timeout=15):
    """
    url_or_urls: str OR list[str]
    tries multiple urls until one works.
    """
    # Normalize to list
    if isinstance(url_or_urls, (list, tuple)):
        urls = [clean_str(u) for u in url_or_urls if clean_str(u) != ""]
    else:
        u = clean_str(url_or_urls)
        urls = [u] if u != "" else []

    if len(urls) == 0:
        return None, None

    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
    }

    for url in urls:
        try:
            r = requests.get(url, timeout=timeout, headers=headers)
            r.raise_for_status()
            img = Image.open(BytesIO(r.content)).convert("RGB")
            return img, url
        except Exception:
            continue

    return None, None

# -----------------------------
# 1) Load + build ASIN-level table
# -----------------------------
print(f"Loading: {INPUT_PATH}")
df = read_any(INPUT_PATH)

needed = [ASIN_COL, BRAND_COL, IMG_URL_COL, TITLE_COL, BENEFIT_COL]
missing_cols = [c for c in needed if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}\nAvailable columns: {df.columns.tolist()}")

# txn_count = how many rows per asin (works even if df is transaction-level)
asin_counts = df.groupby(ASIN_COL).size().rename("txn_count").reset_index()

asin_tbl = (
    df.groupby(ASIN_COL)
      .agg({
          BRAND_COL: "first",
          IMG_URL_COL: first_nonempty,
          TITLE_COL: first_nonempty,
          BENEFIT_COL: first_nonempty
      })
      .reset_index()
      .merge(asin_counts, on=ASIN_COL, how="left")
)

# Apply ASIN image overrides (multi-url)
if len(ASIN_IMAGE_OVERRIDE) > 0:
    n_override = asin_tbl[ASIN_COL].isin(list(ASIN_IMAGE_OVERRIDE.keys())).sum()
    print(f"\nApplying ASIN_IMAGE_OVERRIDE to {n_override} ASIN rows (multi-URL fallback later)...")

# Build combined text
asin_tbl["product_text"] = (
    asin_tbl[TITLE_COL].map(clean_str) + " " + asin_tbl[BENEFIT_COL].map(clean_str)
).str.strip()

# Flags (initial)
asin_tbl["has_text"] = (asin_tbl["product_text"].map(clean_str) != "").astype(int)

print("\nASIN-level table (head):")
print(asin_tbl[[ASIN_COL, BRAND_COL, "txn_count", IMG_URL_COL, "has_text"]].head())
print(f"Unique ASINs: {len(asin_tbl)}")
print(f"Brands: {asin_tbl[BRAND_COL].nunique()}")


# -----------------------------
# 2) Load CLIP
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nLoading CLIP model on {device}: {CLIP_MODEL_NAME}")
model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
model.eval()

D = model.config.projection_dim


# -----------------------------
# 3) Compute CLIP text embeddings (ASIN-level)
# -----------------------------
print("\nComputing CLIP TEXT embeddings (ASIN-level)...")
texts = asin_tbl["product_text"].tolist()
has_text = asin_tbl["has_text"].values.astype(bool)

text_emb = np.zeros((len(texts), D), dtype=np.float32)

BATCH = 64
with torch.no_grad():
    for start in tqdm(range(0, len(texts), BATCH), desc="Text batches"):
        end = min(start + BATCH, len(texts))
        batch_texts = texts[start:end]
        batch_texts_safe = [t if clean_str(t) != "" else " " for t in batch_texts]

        inputs = processor(text=batch_texts_safe, return_tensors="pt", padding=True, truncation=True).to(device)
        feats = model.get_text_features(**inputs)
        feats = feats / feats.norm(dim=1, keepdim=True)
        text_emb[start:end] = feats.detach().cpu().numpy().astype(np.float32)

text_emb[~has_text] = 0.0


# -----------------------------
# 4) Compute CLIP image embeddings (ASIN-level) with override fallback
# -----------------------------
print("\nComputing CLIP IMAGE embeddings (ASIN-level)...")
img_emb = np.zeros((len(asin_tbl), D), dtype=np.float32)
has_img_flag = np.zeros((len(asin_tbl),), dtype=bool)

# For debug: track which URL succeeded
chosen_img_url = [""] * len(asin_tbl)

IMG_BATCH = 16
with torch.no_grad():
    idx = 0
    pbar = tqdm(total=len(asin_tbl), desc="Images")
    while idx < len(asin_tbl):
        batch_indices = list(range(idx, min(idx + IMG_BATCH, len(asin_tbl))))
        images = []
        valid_i = []

        for i in batch_indices:
            asin = clean_str(asin_tbl.loc[i, ASIN_COL])
            base_url = asin_tbl.loc[i, IMG_URL_COL]
            # If asin in overrides -> try those urls first; else try base_url
            url_or_urls = ASIN_IMAGE_OVERRIDE.get(asin, base_url)

            img, used_url = download_image(url_or_urls)
            if img is not None:
                images.append(img)
                valid_i.append(i)
                chosen_img_url[i] = used_url

        if len(images) > 0:
            inputs = processor(images=images, return_tensors="pt").to(device)
            feats = model.get_image_features(**inputs)
            feats = feats / feats.norm(dim=1, keepdim=True)
            feats_np = feats.detach().cpu().numpy().astype(np.float32)

            for k, i in enumerate(valid_i):
                img_emb[i] = feats_np[k]
                has_img_flag[i] = True

        idx += IMG_BATCH
        pbar.update(len(batch_indices))
    pbar.close()

asin_tbl["has_img"] = has_img_flag.astype(int)
asin_tbl["chosen_img_url"] = chosen_img_url

# Quick APAGARD check
apagard_asins = ["B0016GCZQO", "B0016GCZSC", "B0016GHKLO", "B06X3Q9B5X"]
mask_ap = asin_tbl[ASIN_COL].isin(apagard_asins)
if mask_ap.any():
    print("\nAPAGARD image override result:")
    print(asin_tbl.loc[mask_ap, [ASIN_COL, BRAND_COL, "has_img", "chosen_img_url"]].to_string(index=False))


# -----------------------------
# 4.5) PCA + SAVE ASIN-LEVEL FEATURES (for product-level BLP)
# -----------------------------
print("\nRunning PCA at ASIN level (for product-level model)...")

N = len(asin_tbl)

# --- IMAGE PCA: fit only on ASINs that actually have an image ---
valid_img = has_img_flag.astype(bool)
img_pcs_asin = np.full((N, N_PCS), np.nan, dtype=np.float32)

if valid_img.sum() > 2:
    sc_img_asin = StandardScaler()
    img_scaled_valid = sc_img_asin.fit_transform(img_emb[valid_img])
    pca_img_asin = PCA(n_components=N_PCS)
    img_pcs_asin[valid_img] = pca_img_asin.fit_transform(img_scaled_valid).astype(np.float32)
    print("Explained variance (ASIN Image PCs):", np.round(pca_img_asin.explained_variance_ratio_ * 100, 2), "%")
else:
    print("⚠ Not enough valid images for PCA.")

# --- TEXT PCA: fit only on ASINs that actually have text ---
valid_txt = asin_tbl["has_text"].values.astype(bool)
txt_pcs_asin = np.full((N, N_PCS), np.nan, dtype=np.float32)

if valid_txt.sum() > 2:
    sc_txt_asin = StandardScaler()
    txt_scaled_valid = sc_txt_asin.fit_transform(text_emb[valid_txt])
    pca_txt_asin = PCA(n_components=N_PCS)
    txt_pcs_asin[valid_txt] = pca_txt_asin.fit_transform(txt_scaled_valid).astype(np.float32)
    print("Explained variance (ASIN Text PCs):", np.round(pca_txt_asin.explained_variance_ratio_ * 100, 2), "%")
else:
    print("⚠ Not enough valid text for PCA.")

asin_out = asin_tbl[[ASIN_COL, BRAND_COL, "txn_count", "has_img", "has_text", "chosen_img_url"]].copy()

for i in range(N_PCS):
    asin_out[f"asin_img_pc{i+1}"]  = img_pcs_asin[:, i]
    asin_out[f"asin_text_pc{i+1}"] = txt_pcs_asin[:, i]

asin_csv = os.path.join(OUTPUT_DIR, "asin_features_for_blp_clip.csv")
asin_par = os.path.join(OUTPUT_DIR, "asin_features_for_blp_clip.parquet")
asin_out.to_csv(asin_csv, index=False)
asin_out.to_parquet(asin_par, index=False)

print(f"\nSaved: {asin_csv}")
print(f"Saved: {asin_par}")
